# Hyperparameter Tuning Phase 1: EuroBERT-210M## Optuna + Stratified 3-Fold Cross-Validation (Chunk-Modus)Systematische Hyperparameter-Suche fuer EuroBERT-210M auf dem deutschenNachrichtenklassifikations-Datensatz (13 Klassen, Bundestagswahl 2025).**Strategie:**- Optuna TPE-Sampler mit 3-Fold Stratified CV- F1 Macro als Optimierungsmetrik- Keine Modellgewichte gespeichert (nur beste Parameter)- Ergebnisse als Report + Visualisierungen auf Google Drive- SQLite-Backend fuer Crash-Recovery (Colab Pro)- **Chunk-Modus:** N Trials pro Run, danach stoppen. Naechster Run laedt DB und macht weiter.**Voraussetzung:** GPU-Runtime (L4 empfohlen), `HF_TOKEN` in Colab Secrets hinterlegt.**Workflow:**1. `RUN_MODE = "new"` → Neue Study + DB erstellen, N_TRIALS_THIS_RUN Trials ausfuehren2. `RUN_MODE = "continue"` → Bestehende DB laden, N_TRIALS_THIS_RUN weitere Trials anhaengen3. Wiederholen bis gewuenschte Gesamtzahl erreicht

## Run-Konfiguration**Hier vor jedem Run anpassen!**

In [ ]:
# ============================================================# CHUNK-RUN KONFIGURATION — VOR JEDEM RUN ANPASSEN# ============================================================# Modus: "new" = neue Study/DB erstellen#         "continue" = bestehende Study fortsetzenRUN_MODE = "new"# Wie viele Trials in DIESEM Run ausfuehren (egal ob complete/pruned/failed)?N_TRIALS_THIS_RUN = 5# DB-Pfad (nur relevant bei "continue"):#   - None  = automatisch die zuletzt erstellte DB verwenden#   - Pfad  = explizit angeben, z.B. "/content/drive/MyDrive/thesis_reports/performance_reports/eurobert_210m_hpt_phase1_001.db"DB_PATH_OVERRIDE = None# Study-Name (sollte gleich bleiben ueber alle Runs!)STUDY_NAME = "eurobert_210m_hpt_phase1"# Optional: Timeout pro Run in Sekunden (None = kein Limit)OPTUNA_TIMEOUT = Noneprint(f"RUN_MODE:           {RUN_MODE}")print(f"N_TRIALS_THIS_RUN:  {N_TRIALS_THIS_RUN}")print(f"DB_PATH_OVERRIDE:   {DB_PATH_OVERRIDE}")print(f"STUDY_NAME:         {STUDY_NAME}")

In [ ]:
# === SETUP ===import os, sys# Repo klonen / aktualisierenREPO = "/content/news_articles_classification_thesis"if not os.path.exists(REPO):    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}else:    !cd {REPO} && git pull -q# Dependencies (+ optuna, plotly, kaleido fuer HPT)!pip install -q transformers[sentencepiece] datasets huggingface_hub \    scikit-learn matplotlib seaborn tqdm pandas accelerate evaluate \    optuna plotly kaleido# Google Drive mounten (persistente Reports + Optuna DB)from google.colab import drivedrive.mount("/content/drive", force_remount=False)# pipeline_utils importierbar machenPIPELINE_DIR = f"{REPO}/Python/classification_pipeline"if PIPELINE_DIR not in sys.path:    sys.path.insert(0, PIPELINE_DIR)# hpt_utils importierbar machenHPT_DIR = f"{REPO}/Python/classification_pipeline/hyper_parameter_tuning"if HPT_DIR not in sys.path:    sys.path.insert(0, HPT_DIR)import importlibimport pipeline_utils as puimportlib.reload(pu)import hpt_utils as huimportlib.reload(hu)# HuggingFace Loginfrom huggingface_hub import loginfrom google.colab import userdatalogin(token=userdata.get("HF_TOKEN"))# Auto-Shutdown Watchdog: maximale Runtime begrenzen (Sicherheitsnetz fuer Colab Pro)import threading, time as _timeMAX_RUNTIME_HOURS = 7.5def _auto_shutdown():    _time.sleep(MAX_RUNTIME_HOURS * 3600)    print(f"\n[WATCHDOG] Max Runtime ({MAX_RUNTIME_HOURS}h) erreicht. Runtime wird beendet.")    try:        from google.colab import runtime        runtime.unassign()    except Exception:        passthreading.Thread(target=_auto_shutdown, daemon=True).start()print(f"Reports-Ordner: {pu.REPORTS_DIR}")print(f"Setup abgeschlossen. Watchdog: Runtime wird nach max {MAX_RUNTIME_HOURS}h beendet.")

In [ ]:
# ===== HPT CONFIGURATION =====import torchimport numpy as npMODEL_ID = "EuroBERT/EuroBERT-210m"MODEL_SHORT_NAME = "eurobert_210m"MAX_LENGTH = 2048RANDOM_SEED = 42# ----- Cross-Validation -----N_FOLDS = 3# ----- Optuna -----OPTUNA_SEED = 42# ----- Fixed Training Parameters -----FIXED_GRADIENT_CHECKPOINTING = FalseFIXED_LOGGING_STEPS = 10FIXED_REPORT_TO = "tensorboard"FIXED_DATALOADER_NUM_WORKERS = 4FIXED_EARLY_STOPPING_PATIENCE = 3FIXED_GROUP_BY_LENGTH = True# Mixed Precision: GPU-adaptivif torch.cuda.is_available():    _gpu_cap = torch.cuda.get_device_capability()    _gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9    if _gpu_cap[0] >= 8:        FIXED_BF16 = True        FIXED_FP16 = False    else:        FIXED_BF16 = False        FIXED_FP16 = True    FIXED_OPTIM = "adamw_torch_fused"    if _gpu_mem >= 40:        FIXED_BATCH_SIZE_EVAL = 32    elif _gpu_mem >= 20:        FIXED_BATCH_SIZE_EVAL = 16    else:        FIXED_BATCH_SIZE_EVAL = 8    print(f"GPU: {torch.cuda.get_device_name(0)} ({_gpu_mem:.1f} GB, CC {_gpu_cap[0]}.{_gpu_cap[1]})")    print(f"  FP16={FIXED_FP16}, BF16={FIXED_BF16}, Eval Batch={FIXED_BATCH_SIZE_EVAL}")    print(f"  Gradient Checkpointing: {FIXED_GRADIENT_CHECKPOINTING}")else:    raise RuntimeError("HPT benoetigt eine GPU! Bitte Colab Runtime aendern.")# ----- Hyperparameter Search Ranges -----HP_RANGES = {    "learning_rate": (1e-5, 5e-5),    "weight_decay": (0.0, 0.1),    "warmup_ratio": (0.0, 0.15),    "label_smoothing_factor": (0.0, 0.1),    "lr_scheduler_type": ["linear", "cosine"],    "per_device_train_batch_size": [4, 8],    "num_train_epochs": (5, 15),}EFFECTIVE_BATCH_SIZE = 16# ----- Labels -----ALL_LABELS = [    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gef\u00e4lle",    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",    "Gesundheitswesen, Pflege", "Kosten/L\u00f6hne/Preise",    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",]# Split-KonfigurationTEST_PER_CLASS = 30print(f"\nOptuna HPT: {N_TRIALS_THIS_RUN} Trials in diesem Run, {N_FOLDS}-Fold CV")print(f"Modell: {MODEL_ID}")print(f"Max Length: {MAX_LENGTH}")print(f"Effektive Batch Size: {EFFECTIVE_BATCH_SIZE}")print(f"RUN_MODE: {RUN_MODE}")print(f"Labels: {len(ALL_LABELS)} Klassen")

In [ ]:
# ===== DATEN LADEN & CUSTOM SPLIT =====import pandas as pdfrom datasets import load_datasetnp.random.seed(RANDOM_SEED)ds = load_dataset(pu.DATASET_ID)train_hf = ds["train"].to_pandas()test_hf = ds["test"].to_pandas()all_labelled = pd.concat([train_hf, test_hf], ignore_index=True)print(f"Gesamtpool gelabelter Artikel: {len(all_labelled)}")print(f"Klassen im Datensatz: {all_labelled['label'].nunique()}")print()# --- Test-Split (identisch wie im Fine-Tuning Notebook) ---test_indices = []rest_indices = []for label in ALL_LABELS:    label_mask = all_labelled["label"] == label    label_indices = all_labelled[label_mask].index.tolist()    n_total = len(label_indices)    if n_total < 60:        n_test = n_total // 2        print(f"  {label}: nur {n_total} Artikel -> {n_test} fuer Test (Haelfte)")    else:        n_test = TEST_PER_CLASS    np.random.shuffle(label_indices)    test_indices.extend(label_indices[:n_test])    rest_indices.extend(label_indices[n_test:])test_df = all_labelled.loc[test_indices].reset_index(drop=True)cv_pool_df = all_labelled.loc[rest_indices].reset_index(drop=True)print(f"\nTest (eingefroren):  {len(test_df)} Artikel")print(f"CV-Pool (fuer Folds): {len(cv_pool_df)} Artikel")# Klassenverteilungprint("\nCV-Pool Klassenverteilung:")cv_dist = cv_pool_df["label"].value_counts().sort_index()for label, count in cv_dist.items():    print(f"  {label}: {count}")print(f"  TOTAL: {len(cv_pool_df)}")

In [ ]:
# ===== LABEL ENCODING =====label2id = {label: idx for idx, label in enumerate(ALL_LABELS)}id2label = {idx: label for idx, label in enumerate(ALL_LABELS)}cv_pool_df["label_id"] = cv_pool_df["label"].map(label2id)test_df["label_id"] = test_df["label"].map(label2id)assert cv_pool_df["label_id"].isna().sum() == 0, "Unbekannte Labels im CV-Pool!"assert test_df["label_id"].isna().sum() == 0, "Unbekannte Labels im Test-Set!"print("Label-Mapping:")for label, idx in label2id.items():    print(f"  {idx:>2}: {label}")print(f"\nAnzahl Klassen: {len(ALL_LABELS)}")

In [ ]:
# ===== TOKENIZER + EUROBERT ROPE FIX =====from transformers import AutoTokenizerfrom transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONStokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)# EuroBERT ROPE Fixdef _default_rope_init(config, device=None, **kwargs):    base = getattr(config, "rope_theta", 10000.0)    partial_rotary_factor = getattr(config, "partial_rotary_factor", 1.0)    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)    dim = int(head_dim * partial_rotary_factor)    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim))    return inv_freq, 1.0ROPE_INIT_FUNCTIONS["default"] = _default_rope_initprint("ROPE_INIT_FUNCTIONS gepatcht.")def tokenize_fn(examples):    return tokenizer(        examples["text"],        max_length=MAX_LENGTH,        truncation=True,    )print(f"Tokenizer geladen: {MODEL_ID}")

In [ ]:
# ===== OPTUNA OBJECTIVE MIT K-FOLD CV =====import gcimport shutilimport optunafrom sklearn.model_selection import StratifiedKFoldfrom datasets import Datasetfrom transformers import (    AutoModelForSequenceClassification,    AutoConfig,    TrainingArguments,    Trainer,    EarlyStoppingCallback,    DataCollatorWithPadding,)data_collator = DataCollatorWithPadding(tokenizer=tokenizer)compute_metrics = hu.make_compute_metrics(ALL_LABELS, id2label)# Fold-Indizes vorab berechnen (gleiche Folds fuer jeden Trial -> fairer Vergleich)skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)fold_indices = list(skf.split(cv_pool_df, cv_pool_df["label_id"]))print(f"Stratified {N_FOLDS}-Fold CV:")for i, (train_idx, val_idx) in enumerate(fold_indices):    val_labels = cv_pool_df.iloc[val_idx]["label"].value_counts()    min_val_count = val_labels.min()    min_val_class = val_labels.idxmin()    print(f"  Fold {i+1}: Train={len(train_idx)}, Val={len(val_idx)}, "          f"min Val-Klasse: {min_val_class} ({min_val_count} Samples)")def create_model(seed=42):    torch.manual_seed(seed)    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)    config.num_labels = len(ALL_LABELS)    config.id2label = id2label    config.label2id = label2id    model = AutoModelForSequenceClassification.from_pretrained(        MODEL_ID,        config=config,        ignore_mismatched_sizes=True,        trust_remote_code=True,    )    import torch.nn as nn    for name, module in model.named_modules():        if name in ("dense", "classifier") and isinstance(module, nn.Linear):            nn.init.normal_(module.weight, mean=0.0, std=0.002)            if module.bias is not None:                nn.init.zeros_(module.bias)    if FIXED_GRADIENT_CHECKPOINTING and torch.cuda.is_available():        model.gradient_checkpointing_enable()    return modeldef _cleanup_cuda():    if torch.cuda.is_available():        torch.cuda.empty_cache()    gc.collect()def objective(trial):    # --- Hyperparameter samplen ---    lr = trial.suggest_float("learning_rate", *HP_RANGES["learning_rate"], log=True)    wd = trial.suggest_float("weight_decay", *HP_RANGES["weight_decay"])    warmup = trial.suggest_float("warmup_ratio", *HP_RANGES["warmup_ratio"])    label_smooth = trial.suggest_float("label_smoothing_factor", *HP_RANGES["label_smoothing_factor"])    scheduler = trial.suggest_categorical("lr_scheduler_type", HP_RANGES["lr_scheduler_type"])    batch_size = trial.suggest_categorical("per_device_train_batch_size", HP_RANGES["per_device_train_batch_size"])    epochs = trial.suggest_int("num_train_epochs", *HP_RANGES["num_train_epochs"])    grad_accum = max(1, EFFECTIVE_BATCH_SIZE // batch_size)    print(f"\n{'='*60}")    print(f"Trial {trial.number}: lr={lr:.2e}, wd={wd:.3f}, warmup={warmup:.3f}, "          f"ls={label_smooth:.3f}, sched={scheduler}, "          f"batch={batch_size}, grad_accum={grad_accum}, epochs={epochs}")    print(f"{'='*60}")    fold_scores = []    for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):        print(f"  Fold {fold_idx + 1}/{N_FOLDS}...", end=" ", flush=True)        _cleanup_cuda()        fold_train_df = cv_pool_df.iloc[train_idx]        fold_val_df = cv_pool_df.iloc[val_idx]        train_ds = Dataset.from_pandas(            fold_train_df[["text", "label_id"]].rename(columns={"label_id": "labels"})        )        val_ds = Dataset.from_pandas(            fold_val_df[["text", "label_id"]].rename(columns={"label_id": "labels"})        )        train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])        val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])        train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])        val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])        fold_output_dir = f"/content/hpt_tmp/trial_{trial.number}_fold_{fold_idx}"        fold_logging_dir = f"/content/hpt_tmp/tb_logs/trial_{trial.number:03d}_fold_{fold_idx}"        metrics_callback = hu.HPTMetricsCallback(            trial_number=trial.number,            fold_idx=fold_idx,            all_labels=ALL_LABELS,            id2label=id2label,        )        training_args = TrainingArguments(            output_dir=fold_output_dir,            num_train_epochs=epochs,            learning_rate=lr,            weight_decay=wd,            warmup_ratio=warmup,            label_smoothing_factor=label_smooth,            lr_scheduler_type=scheduler,            per_device_train_batch_size=batch_size,            per_device_eval_batch_size=FIXED_BATCH_SIZE_EVAL,            gradient_accumulation_steps=grad_accum,            bf16=FIXED_BF16,            fp16=FIXED_FP16,            gradient_checkpointing=FIXED_GRADIENT_CHECKPOINTING,            optim=FIXED_OPTIM,            group_by_length=FIXED_GROUP_BY_LENGTH,            eval_strategy="epoch",            save_strategy="epoch",            save_total_limit=1,            load_best_model_at_end=True,            metric_for_best_model="f1_macro",            greater_is_better=True,            logging_strategy="steps",            logging_steps=FIXED_LOGGING_STEPS,            logging_dir=fold_logging_dir,            report_to=FIXED_REPORT_TO,            seed=RANDOM_SEED + fold_idx,            dataloader_num_workers=FIXED_DATALOADER_NUM_WORKERS,            dataloader_pin_memory=True,            disable_tqdm=False,        )        try:            model = create_model(seed=RANDOM_SEED + trial.number * 100 + fold_idx)            trainer = Trainer(                model=model,                args=training_args,                train_dataset=train_ds,                eval_dataset=val_ds,                data_collator=data_collator,                compute_metrics=compute_metrics,                callbacks=[                    EarlyStoppingCallback(early_stopping_patience=FIXED_EARLY_STOPPING_PATIENCE),                    metrics_callback,                ],            )            trainer.train()            if metrics_callback.nan_detected or metrics_callback.lr_zero_detected:                reason = "NaN" if metrics_callback.nan_detected else "LR=0"                print(f"{reason} detected — pruning Trial {trial.number}")                hu.store_fold_metrics_partial(trial, fold_idx, metrics_callback, ALL_LABELS)                trial.set_user_attr("nan_trial", True)                trial.set_user_attr("nan_reason", reason)                trial.set_user_attr("nan_fold_idx", fold_idx)                raise optuna.TrialPruned()            eval_result = trainer.evaluate()            fold_f1 = eval_result["eval_f1_macro"]            if np.isnan(eval_result.get("eval_loss", 0)) or fold_f1 < 0.05:                print(f"NaN/collapse in final eval (F1={fold_f1:.4f}) — pruning")                hu.store_fold_metrics(trial, fold_idx, metrics_callback, eval_result, ALL_LABELS, id2label)                trial.set_user_attr("nan_trial", True)                trial.set_user_attr("nan_reason", "eval_nan")                trial.set_user_attr("nan_fold_idx", fold_idx)                raise optuna.TrialPruned()            hu.store_fold_metrics(trial, fold_idx, metrics_callback, eval_result, ALL_LABELS, id2label)            fold_scores.append(fold_f1)            print(f"F1 Macro = {fold_f1:.4f}")        except torch.cuda.OutOfMemoryError:            print(f"\n  OOM in Trial {trial.number}, Fold {fold_idx + 1}! Ueberspringe Trial.")            _cleanup_cuda()            if os.path.exists(fold_output_dir):                shutil.rmtree(fold_output_dir, ignore_errors=True)            raise optuna.TrialPruned()        finally:            for var in ["trainer", "model", "training_args", "train_ds", "val_ds"]:                try:                    exec(f"del {var}")                except NameError:                    pass            _cleanup_cuda()            if os.path.exists(fold_output_dir):                shutil.rmtree(fold_output_dir, ignore_errors=True)        trial.report(np.mean(fold_scores), fold_idx)        if trial.should_prune():            print(f"  Trial {trial.number} PRUNED nach Fold {fold_idx + 1}")            raise optuna.TrialPruned()    hu.store_trial_summary(trial, N_FOLDS, ALL_LABELS)    mean_f1 = np.mean(fold_scores)    std_f1 = np.std(fold_scores)    print(f"\n  -> Trial {trial.number}: F1 Macro = {mean_f1:.4f} +/- {std_f1:.4f}")    return mean_f1print("Objective Function definiert.")print(f"Folds: {N_FOLDS}")print(f"Effektive Batch Size: {EFFECTIVE_BATCH_SIZE}")

## Study starten / fortsetzen (Chunk-Modus)Diese Zelle ist das Herzstück des Chunk-Modus:- Bei `"new"`: Erstellt neue DB + Study, fuehrt N_TRIALS_THIS_RUN Trials aus- Bei `"continue"`: Laedt bestehende DB, fuehrt N_TRIALS_THIS_RUN **weitere** Trials aus

In [ ]:
# ===== OPTUNA STUDY — CHUNK-MODUS =====from optuna.samplers import TPESamplerfrom optuna.pruners import MedianPrunerfrom pathlib import Pathimport glob# --- DB-Pfad bestimmen ---DB_BASE_DIR = Path(pu.REPORTS_DIR)LATEST_DB_FILE = DB_BASE_DIR / ".latest_hpt_db.txt"if RUN_MODE == "new":    # Neue DB mit Auto-Nummerierung erstellen    existing_dbs = sorted(glob.glob(str(DB_BASE_DIR / f"{STUDY_NAME}_*.db")))    if existing_dbs:        # Hoechste Nummer finden und +1        last_num = 0        for db in existing_dbs:            try:                num = int(Path(db).stem.split("_")[-1])                last_num = max(last_num, num)            except ValueError:                pass        db_number = last_num + 1    else:        db_number = 1    db_filename = f"{STUDY_NAME}_{db_number:03d}.db"    storage_path = str(DB_BASE_DIR / db_filename)    storage_url = f"sqlite:///{storage_path}"    print(f"[NEW] Erstelle neue DB: {storage_path}")elif RUN_MODE == "continue":    if DB_PATH_OVERRIDE:        storage_path = DB_PATH_OVERRIDE    elif LATEST_DB_FILE.exists():        storage_path = LATEST_DB_FILE.read_text().strip()    else:        # Fallback: neueste DB im Ordner finden        existing_dbs = sorted(glob.glob(str(DB_BASE_DIR / f"{STUDY_NAME}_*.db")))        if not existing_dbs:            raise FileNotFoundError(                f"Keine bestehende DB gefunden in {DB_BASE_DIR}! "                f"Bitte RUN_MODE='new' setzen oder DB_PATH_OVERRIDE angeben."            )        storage_path = existing_dbs[-1]        print(f"[AUTO] Neueste DB gefunden: {storage_path}")    if not os.path.exists(storage_path):        raise FileNotFoundError(f"DB nicht gefunden: {storage_path}")    storage_url = f"sqlite:///{storage_path}"    print(f"[CONTINUE] Lade bestehende DB: {storage_path}")else:    raise ValueError(f"Ungueltiger RUN_MODE: {RUN_MODE}. Erlaubt: 'new', 'continue'")# Letzte DB merken fuer naechsten "continue" RunLATEST_DB_FILE.write_text(storage_path)print(f"  -> Gespeichert als letzte DB in: {LATEST_DB_FILE}")# --- Sampler + Pruner ---sampler = TPESampler(seed=OPTUNA_SEED, n_startup_trials=5)pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=1)# --- Study erstellen oder laden ---if RUN_MODE == "new":    study = optuna.create_study(        study_name=STUDY_NAME,        storage=storage_url,        direction="maximize",        sampler=sampler,        pruner=pruner,        load_if_exists=False,    )    print(f"Neue Study erstellt: {STUDY_NAME}")else:    study = optuna.load_study(        study_name=STUDY_NAME,        storage=storage_url,        sampler=sampler,        pruner=pruner,    )    n_existing = len(study.trials)    n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])    n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])    print(f"Study geladen: {n_existing} bestehende Trials ({n_complete} complete, {n_pruned} pruned)")# --- Status vor dem Run ---n_before = len(study.trials)print(f"\n{'='*60}")print(f"Trials vor diesem Run: {n_before}")print(f"Trials in diesem Run:  {N_TRIALS_THIS_RUN}")print(f"{'='*60}")# --- Trials ausfuehren ---timer_hpt = pu.ExperimentTimer()with timer_hpt:    study.optimize(        objective,        n_trials=N_TRIALS_THIS_RUN,        timeout=OPTUNA_TIMEOUT,        gc_after_trial=True,    )n_after = len(study.trials)print(f"\nChunk abgeschlossen: {timer_hpt.duration_formatted}")print(f"Trials ausgefuehrt in diesem Run: {n_after - n_before}")print(f"Trials insgesamt in der DB: {n_after}")# --- Zusammenfassung ---all_trials = study.trialsn_complete_total = len([t for t in all_trials if t.state == optuna.trial.TrialState.COMPLETE])n_pruned_total = len([t for t in all_trials if t.state == optuna.trial.TrialState.PRUNED])n_failed_total = len([t for t in all_trials if t.state == optuna.trial.TrialState.FAIL])print(f"\n{'='*60}")print(f"ZUSAMMENFASSUNG")print(f"{'='*60}")print(f"  Complete: {n_complete_total}")print(f"  Pruned:   {n_pruned_total}")print(f"  Failed:   {n_failed_total}")print(f"  Gesamt:   {n_after}")if n_complete_total > 0:    print(f"\n  Bester Trial:      {study.best_trial.number}")    print(f"  Bester F1 Macro:   {study.best_value:.4f}")    print(f"\n  Beste Hyperparameter:")    for key, val in study.best_params.items():        print(f"    {key}: {val}")else:    print("\n  Noch keine abgeschlossenen Trials — weitere Runs noetig.")print(f"\nDB: {storage_path}")print(f"\nNaechster Schritt: RUN_MODE='continue' setzen und erneut ausfuehren,")print(f"oder mit den Analyse-Zellen unten fortfahren.")print(f"{'='*60}")

In [ ]:
# ===== OPTUNA QUICK SANITY CHECK =====import optuna.visualization as visif n_complete_total >= 2:    fig_history = vis.plot_optimization_history(study)    fig_history.update_layout(title="Optimization History: F1 Macro ueber Trials")    fig_history.show()    try:        fig_importance = vis.plot_param_importances(study)        fig_importance.update_layout(title="Parameter Importance (fANOVA)")        fig_importance.show()    except Exception as e:        print(f"Parameter Importance nicht verfuegbar: {e}")    print("Sanity-Check Plots erstellt.")else:    print(f"Nur {n_complete_total} abgeschlossene Trials — mindestens 2 noetig fuer Plots.")    print("Weitere Runs mit RUN_MODE='continue' ausfuehren.")

In [ ]:
# ===== QUICK RESULTS OVERVIEW =====trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state", "duration"))trials_df = trials_df.sort_values("value", ascending=False)completed = trials_df[trials_df["state"] == "COMPLETE"].copy()pruned = trials_df[trials_df["state"] == "PRUNED"].copy()print(f"Trials: {len(completed)} abgeschlossen, {len(pruned)} gepruned")if len(completed) > 0:    print(f"\nTop 5 Trials:")    top5_cols = ["number", "value"] + [c for c in completed.columns if c.startswith("params_")]    print(completed[top5_cols].head().to_string(index=False))    print(f"\nStatistiken ueber alle abgeschlossenen Trials:")    print(f"  Mean F1:   {completed['value'].mean():.4f}")    print(f"  Std F1:    {completed['value'].std():.4f}")    print(f"  Min F1:    {completed['value'].min():.4f}")    print(f"  Max F1:    {completed['value'].max():.4f}")else:    print("Noch keine abgeschlossenen Trials.")

## Report generieren (optional)Nur ausfuehren wenn die gewuenschte Anzahl Trials erreicht ist.

In [ ]:
# ===== HPT REPORT GENERIEREN =====import jsonfrom datetime import datetimeif n_complete_total == 0:    print("Keine abgeschlossenen Trials — Report kann nicht erstellt werden.")else:    now = datetime.now()    report_name = f"{now.strftime('%d%m%y')}_{MODEL_SHORT_NAME}_hpt_phase1"    duration_str = timer_hpt.duration_formatted if 'timer_hpt' in dir() else "N/A (aus DB geladen)"    # --- Markdown Report ---    report_lines = [        f"# HPT Report: EuroBERT-210M Phase 1",        f"**Generated:** {now.strftime('%Y-%m-%d %H:%M:%S')}",        "",        "---",        "",        "## Configuration",        "| Property | Value |",        "|---|---|",        f"| Model | {MODEL_ID} |",        f"| Total Trials | {len(study.trials)} |",        f"| N Folds | {N_FOLDS} |",        f"| Completed Trials | {len(completed)} |",        f"| Pruned Trials | {len(pruned)} |",        f"| Last Run Duration | {duration_str} |",        f"| GPU | {pu.get_gpu_info()['gpu_name']} |",        f"| CV Pool Size | {len(cv_pool_df)} |",        f"| Test Size (frozen) | {len(test_df)} |",        f"| Effective Batch Size | {EFFECTIVE_BATCH_SIZE} |",        f"| DB Path | {storage_path} |",        "",        "## Fixed Parameters",        "| Parameter | Value |",        "|---|---|",        f"| bf16 | {FIXED_BF16} |",        f"| fp16 | {FIXED_FP16} |",        f"| gradient_checkpointing | {FIXED_GRADIENT_CHECKPOINTING} |",        f"| group_by_length | {FIXED_GROUP_BY_LENGTH} |",        f"| optim | {FIXED_OPTIM} |",        f"| early_stopping_patience | {FIXED_EARLY_STOPPING_PATIENCE} |",        f"| logging_steps | {FIXED_LOGGING_STEPS} |",        f"| max_length | {MAX_LENGTH} |",        "",        "## Search Ranges",        "| Parameter | Range |",        "|---|---|",    ]    for param, range_val in HP_RANGES.items():        report_lines.append(f"| {param} | {range_val} |")    report_lines += [        "",        f"**Best F1 Macro (CV Mean): {study.best_value:.4f}**",        f"**Best Trial: {study.best_trial.number}**",        "",        "## Alle Trials",        "",    ]    table_cols = ["number", "value", "state"] + [c for c in trials_df.columns if c.startswith("params_")]    report_lines.append(trials_df[table_cols].to_markdown(index=False))    report_lines += [        "",        "---",        "*Best parameter extraction: see analyse_hyperparameter_tuning.ipynb*",        f"*Generated by eurobert_210_hpt_phase_1.ipynb (chunk mode)*",    ]    report_md = "\n".join(report_lines)    report_path = Path(pu.REPORTS_DIR) / f"{report_name}.md"    report_path.write_text(report_md, encoding="utf-8")    # --- JSON Sidecar ---    json_data = {        "report_id": report_name,        "model_id": MODEL_ID,        "total_trials": len(study.trials),        "n_folds": N_FOLDS,        "best_value": round(study.best_value, 4),        "best_trial_number": study.best_trial.number,        "best_params": study.best_params,        "all_trials": [            {                "number": t.number,                "value": round(t.value, 4) if t.value is not None else None,                "params": t.params,                "state": str(t.state.name),                "duration_s": round(t.duration.total_seconds(), 1) if t.duration else None,            }            for t in study.trials        ],        "search_ranges": {k: str(v) for k, v in HP_RANGES.items()},        "fixed_params": {            "bf16": FIXED_BF16,            "fp16": FIXED_FP16,            "gradient_checkpointing": FIXED_GRADIENT_CHECKPOINTING,            "optim": FIXED_OPTIM,            "early_stopping_patience": FIXED_EARLY_STOPPING_PATIENCE,            "group_by_length": FIXED_GROUP_BY_LENGTH,            "max_length": MAX_LENGTH,            "effective_batch_size": EFFECTIVE_BATCH_SIZE,        },        "last_run_duration": duration_str,        "gpu": pu.get_gpu_info(),        "db_path": storage_path,    }    json_path = Path(pu.REPORTS_DIR) / f"{report_name}.json"    json_path.write_text(json.dumps(json_data, ensure_ascii=False, indent=2), encoding="utf-8")    print(f"Report: {report_path}")    print(f"JSON:   {json_path}")

In [ ]:
# ===== HINWEIS: PARAMETER-EXTRAKTION =====print("Best parameter extraction: see analyse_hyperparameter_tuning.ipynb")print(f"DB location: {storage_path}")

In [ ]:
# ===== CLEANUP + AUTO-SHUTDOWN =====import shutilif os.path.exists("/content/hpt_tmp"):    for item in Path("/content/hpt_tmp").iterdir():        if item.name != "tb_logs":            shutil.rmtree(item, ignore_errors=True)    print("Temporaere Checkpoint-Dateien geloescht.")    print("TensorBoard Logs behalten: /content/hpt_tmp/tb_logs/")if torch.cuda.is_available():    torch.cuda.empty_cache()    gc.collect()    free_mem = torch.cuda.mem_get_info()[0] / 1e9    print(f"GPU VRAM frei: {free_mem:.1f} GB")print(f"\nErgebnisse auf Google Drive:")print(f"  DB: {storage_path}")if 'report_path' in dir():    print(f"  Report: {report_path}")    print(f"  JSON:   {json_path}")# Runtime beenden (Colab Pro — spart Kosten)# Auskommentieren wenn man weitere Runs machen will!# print("\nRuntime wird beendet...")# from google.colab import runtime# runtime.unassign()print("\nFertig! Runtime bleibt aktiv fuer weitere Runs.")print("Fuer naechsten Run: RUN_MODE='continue' setzen und Zellen erneut ausfuehren.")